# 03_error_analysis

Notebook này dùng để **phân tích lỗi của baseline lookup** sau khi bạn đã chạy xong `02_baseline_lookup.ipynb`.

## Mục tiêu
- Đọc lại các file lỗi đã lưu
- Xem lookup đang sai ở token nào nhiều nhất
- Xem lookup hay đoán nhầm kiểu gì
- Tách lỗi theo một số nhóm đơn giản:
  - OOV
  - token mơ hồ
  - token không cần normalize nhưng lookup lại sửa sai
  - token cần normalize nhưng lookup vẫn chưa sửa đúng
- Chọn hướng phát triển tiếp theo:
  - thêm rules
  - chia easy / hard token
  - chuẩn bị cho hybrid system

## 1. Import thư viện

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from collections import Counter, defaultdict

## 2. Đường dẫn file

In [ ]:
train_path = "../data/multilexnorm2026_vi_train.csv"
val_path   = "../data/multilexnorm2026_vi_validation.csv"

all_results_path      = "../outputs/val_lookup_all_results.csv"
wrong_sentences_path  = "../outputs/val_lookup_wrong_sentences.csv"
token_results_path    = "../outputs/val_lookup_token_level_results.csv"
wrong_tokens_path     = "../outputs/val_lookup_wrong_tokens.csv"

## 3. Đọc dữ liệu gốc và file lỗi

In [ ]:
train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)

all_results_df     = pd.read_csv(all_results_path)
wrong_sentence_df  = pd.read_csv(wrong_sentences_path)
token_results_df   = pd.read_csv(token_results_path)
wrong_token_df     = pd.read_csv(wrong_tokens_path)

print("train_df shape        :", train_df.shape)
print("val_df shape          :", val_df.shape)
print("all_results_df shape  :", all_results_df.shape)
print("wrong_sentence_df     :", wrong_sentence_df.shape)
print("token_results_df      :", token_results_df.shape)
print("wrong_token_df        :", wrong_token_df.shape)

## 4. Parse lại token list từ string

In [ ]:
def parse_token_list(text):
    text = str(text)
    tokens = re.findall(r"'([^']*)'", text)
    return tokens

for df in [train_df, val_df]:
    df["raw_tokens"] = df["raw"].apply(parse_token_list)
    df["norm_tokens"] = df["norm"].apply(parse_token_list)

for col in ["raw_tokens", "gold_tokens", "pred_tokens"]:
    if col in wrong_sentence_df.columns:
        wrong_sentence_df[col] = wrong_sentence_df[col].apply(parse_token_list)

print("Done parsing token columns.")

## 5. Xem nhanh một vài câu sai

In [ ]:
display(wrong_sentence_df.head(10))

## 6. Top token bị lookup đoán sai nhiều nhất

In [ ]:
raw_error_counts = (
    wrong_token_df["raw_token"]
    .value_counts()
    .reset_index()
)
raw_error_counts.columns = ["raw_token", "error_count"]

display(raw_error_counts.head(30))

### Cách đọc
Nếu một token xuất hiện nhiều ở bảng này, điều đó thường có nghĩa là:
- token đó mơ hồ
- hoặc token đó là OOV/hiếm
- hoặc token đó có nhiều biến thể khó xử lý bằng lookup

## 7. Top pattern lỗi phổ biến nhất

In [ ]:
error_pattern_df = (
    wrong_token_df
    .groupby(["raw_token", "gold_token", "pred_token"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(error_pattern_df.head(50))

## 8. Token sai nào là OOV

In [ ]:
oov_stats = (
    wrong_token_df["is_oov_vs_train"]
    .value_counts()
    .sort_index()
    .reset_index()
)
oov_stats.columns = ["is_oov_vs_train", "count"]
display(oov_stats)

num_wrong_tokens = len(wrong_token_df)
num_wrong_oov = int((wrong_token_df["is_oov_vs_train"] == 1).sum())
oov_ratio = num_wrong_oov / num_wrong_tokens if num_wrong_tokens > 0 else 0

print("Wrong OOV tokens:", num_wrong_oov)
print("Wrong-token OOV ratio:", round(oov_ratio * 100, 2), "%")

## 9. Lỗi theo 2 nhóm lớn

In [ ]:
need_norm_wrong_df = wrong_token_df[wrong_token_df["needs_normalization"] == 1].copy()
over_change_df     = wrong_token_df[wrong_token_df["needs_normalization"] == 0].copy()

print("Wrong tokens that truly need normalization :", len(need_norm_wrong_df))
print("Wrong tokens that should have been kept     :", len(over_change_df))

if len(wrong_token_df) > 0:
    print("Ratio need-normalization wrong:",
          round(len(need_norm_wrong_df) / len(wrong_token_df) * 100, 2), "%")
    print("Ratio over-change wrong:",
          round(len(over_change_df) / len(wrong_token_df) * 100, 2), "%")

### 9.1. Top token cần normalize nhưng lookup vẫn sai

In [ ]:
need_norm_counts = (
    need_norm_wrong_df["raw_token"]
    .value_counts()
    .reset_index()
)
need_norm_counts.columns = ["raw_token", "wrong_count"]

display(need_norm_counts.head(30))

### 9.2. Top token không cần đổi nhưng lookup lại đổi sai

In [ ]:
over_change_counts = (
    over_change_df["raw_token"]
    .value_counts()
    .reset_index()
)
over_change_counts.columns = ["raw_token", "wrong_count"]

display(over_change_counts.head(30))

## 10. Tìm token mơ hồ từ train

In [ ]:
pair_counter = Counter()
raw_to_norms = defaultdict(Counter)

for raw_tokens, norm_tokens in zip(train_df["raw_tokens"], train_df["norm_tokens"]):
    if len(raw_tokens) != len(norm_tokens):
        continue
    for raw_tok, norm_tok in zip(raw_tokens, norm_tokens):
        pair_counter[(raw_tok, norm_tok)] += 1
        raw_to_norms[raw_tok][norm_tok] += 1

ambiguous_info = []
for raw_tok, counter in raw_to_norms.items():
    total_count = sum(counter.values())
    top1_norm, top1_count = counter.most_common(1)[0]
    num_choices = len(counter)
    confidence = top1_count / total_count if total_count > 0 else 0

    ambiguous_info.append({
        "raw_token": raw_tok,
        "num_choices": num_choices,
        "total_count": total_count,
        "top1_norm": top1_norm,
        "top1_count": top1_count,
        "lookup_confidence": confidence
    })

ambiguous_df = pd.DataFrame(ambiguous_info)
ambiguous_df = ambiguous_df.sort_values(
    by=["num_choices", "total_count"],
    ascending=[False, False]
)

display(ambiguous_df.head(30))

## 11. Ghép thông tin ambiguity vào token lỗi

In [ ]:
wrong_token_with_meta_df = wrong_token_df.merge(
    ambiguous_df[["raw_token", "num_choices", "total_count", "top1_norm", "lookup_confidence"]],
    on="raw_token",
    how="left"
)

wrong_token_with_meta_df["num_choices"] = wrong_token_with_meta_df["num_choices"].fillna(0)
wrong_token_with_meta_df["total_count"] = wrong_token_with_meta_df["total_count"].fillna(0)
wrong_token_with_meta_df["lookup_confidence"] = wrong_token_with_meta_df["lookup_confidence"].fillna(0.0)

display(wrong_token_with_meta_df.head(10))

## 12. Token sai nào vừa mơ hồ vừa có confidence thấp

In [ ]:
hard_token_candidates_df = wrong_token_with_meta_df[
    (wrong_token_with_meta_df["num_choices"] > 1) &
    (wrong_token_with_meta_df["lookup_confidence"] < 0.95)
].copy()

hard_token_counts = (
    hard_token_candidates_df["raw_token"]
    .value_counts()
    .reset_index()
)
hard_token_counts.columns = ["raw_token", "error_count"]

display(hard_token_counts.head(30))

print("Number of wrong tokens that are ambiguous + low-confidence:",
      len(hard_token_candidates_df))

## 13. Xem top câu sai nặng nhất

In [ ]:
display(wrong_sentence_df.head(20))

## 14. In ra vài câu khó nhất theo dạng dễ đọc

In [ ]:
top_wrong_sent_ids = wrong_sentence_df["sent_idx"].head(10).tolist()

for sent_id in top_wrong_sent_ids:
    row = wrong_sentence_df[wrong_sentence_df["sent_idx"] == sent_id].iloc[0]
    print("=" * 100)
    print("Sentence index:", sent_id)
    print("RAW :", row["raw_tokens"])
    print("GOLD:", row["gold_tokens"])
    print("PRED:", row["pred_tokens"])
    print("Token errors:", row["token_errors"])
    print("Error positions:", row["error_positions"])

## 15. Tạo bảng tổng hợp các nhóm lỗi

In [ ]:
def classify_error(row):
    if row["is_oov_vs_train"] == 1:
        return "oov"
    if row["needs_normalization"] == 0:
        return "over_change"
    if row["num_choices"] > 1 and row["lookup_confidence"] < 0.95:
        return "ambiguous_low_conf"
    if row["total_count"] > 0 and row["total_count"] < 5:
        return "rare_seen_token"
    return "other_need_norm_error"

wrong_token_with_meta_df["error_group"] = wrong_token_with_meta_df.apply(classify_error, axis=1)

error_group_counts = (
    wrong_token_with_meta_df["error_group"]
    .value_counts()
    .reset_index()
)
error_group_counts.columns = ["error_group", "count"]

display(error_group_counts)

## 16. Xem ví dụ cho từng nhóm lỗi

In [ ]:
for group_name in wrong_token_with_meta_df["error_group"].value_counts().index.tolist():
    print("\n" + "=" * 100)
    print("ERROR GROUP:", group_name)
    sample_df = wrong_token_with_meta_df[wrong_token_with_meta_df["error_group"] == group_name].head(10)
    display(sample_df[[
        "sent_idx", "token_pos", "raw_token", "gold_token", "pred_token",
        "is_oov_vs_train", "needs_normalization", "num_choices",
        "total_count", "lookup_confidence"
    ]])

## 17. Lưu các bảng phân tích ra file

In [ ]:
os.makedirs("../outputs", exist_ok=True)

raw_error_counts.to_csv("../outputs/error_top_raw_tokens.csv", index=False, encoding="utf-8-sig")
error_pattern_df.to_csv("../outputs/error_top_patterns.csv", index=False, encoding="utf-8-sig")
wrong_token_with_meta_df.to_csv("../outputs/error_wrong_tokens_with_meta.csv", index=False, encoding="utf-8-sig")
error_group_counts.to_csv("../outputs/error_group_summary.csv", index=False, encoding="utf-8-sig")
hard_token_counts.to_csv("../outputs/error_hard_token_candidates.csv", index=False, encoding="utf-8-sig")

print("Saved:")
print("- ../outputs/error_top_raw_tokens.csv")
print("- ../outputs/error_top_patterns.csv")
print("- ../outputs/error_wrong_tokens_with_meta.csv")
print("- ../outputs/error_group_summary.csv")
print("- ../outputs/error_hard_token_candidates.csv")

## 18. Kết luận cuối notebook

### Những câu hỏi cần trả lời
- Lookup đang sai chủ yếu vì OOV, ambiguity hay over-correction?
- Những token nào là token khó rõ ràng nhất?
- Có nhóm lỗi nào có thể sửa bằng rule đơn giản không?
- Có nên bắt đầu chia easy / hard token chưa?
- Nếu làm hybrid, nhóm token nào nên đưa sang model trước?

### Gợi ý kết luận
- Nếu lỗi tập trung vào token mơ hồ như `t`, `m`, `n`, ... thì nên cân nhắc **context-aware model** hoặc ít nhất là **đừng sửa bừa bằng lookup**.
- Nếu lỗi tập trung vào OOV/biến thể lạ, có thể thử **rules** hoặc **character-level handling** trước.
- Nếu có nhiều lỗi over-change, nên thiết kế cơ chế **lookup confidence** để chỉ sửa khi đủ chắc.